# Truncate All User Tables

This notebook truncates (removes all data from) all user tables while preserving table structures.
- Automatically discovers all user schemas and tables
- Skips system schemas

In [ ]:
# === TRUNCATE ALL USER TABLES ===
def truncate_all_user_tables():
    """Truncate all user tables (excludes system schemas)"""
    
    try:
        # Get all schemas, excluding system ones
        all_schemas = spark.sql("SHOW DATABASES").collect()
        system_schemas = {'default', 'information_schema', 'sys', 'hive_metastore'}
        user_schemas = [schema[0] for schema in all_schemas if schema[0] not in system_schemas]
        
        if not user_schemas:
            print("✅ No user schemas found!")
            return 0
        
        print(f"🔍 Found schemas: {', '.join(user_schemas)}")
        
        total_truncated = 0
        for schema in user_schemas:
            try:
                # Get all tables in schema
                tables = spark.sql(f"SHOW TABLES IN {schema}").collect()
                table_names = [table.tableName for table in tables]
                
                if not table_names:
                    print(f"📭 No tables in {schema}")
                    continue
                
                print(f"\n🔨 Truncating {len(table_names)} tables in '{schema}'...")
                
                for table_name in table_names:
                    full_table = f"{schema}.{table_name}"
                    try:
                        spark.sql(f"TRUNCATE TABLE {full_table}")
                        print(f"  ✅ {full_table}")
                        total_truncated += 1
                    except Exception as e:
                        print(f"  ❌ {full_table}: {e}")
                        
            except Exception as e:
                print(f"⚠️ Error processing schema {schema}: {e}")
        
        print(f"\n🎉 Completed! {total_truncated} tables truncated.")
        return total_truncated
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return 0

# === EXECUTE ===
print("🚨 TRUNCATING ALL USER TABLES")
truncate_all_user_tables()
print("💡 Table structures preserved, only data removed!")